In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# 設定 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.get_device_name(0)}")
print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

CUDA available: True
Current device: NVIDIA GeForce RTX 5070
Compute Capability: (12, 0)


In [ ]:
#### TODO 讀取紅酒與白酒訓練資料
data = pd.read_csv('winequality-red_train.csv', delimiter=',')
#### TODO
selected_features = True
if selected_features:
  #### TODO 篩選酒類所需特徵，並進行訓練
  selected_features = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
  #### TODO
if selected_features:
  data = data[selected_features + ['quality']]

# 分割特徵與標籤
X = data.iloc[:, :-1].values  # 特徵
y = data.iloc[:, -1].values   # 標籤 (Wine Quality)

# 標準化數據
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [14]:
# 切分訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 轉換為 PyTorch Tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

# 創建 DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [15]:
# 定義神經網絡模型
class WineQualityNN(nn.Module):
    def __init__(self, input_dim):
        super(WineQualityNN, self).__init__()
        #### TODO 類別確認
        self.fc1 = nn.Linear(input_dim, 128)
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(64, 11)  # Wine quality 範圍為 0-10
        #### TODO

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout1(x)
        x = self.fc3(x)
        return x

In [16]:
# 初始化模型
input_dim = X.shape[1]
model = WineQualityNN(input_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [17]:
# 訓練模型
def train_model(model, train_loader, criterion, optimizer, epochs=20):
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

In [18]:
# 測試模型
def test_model(model, test_loader):
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [19]:
# 執行訓練與測試
train_model(model, train_loader, criterion, optimizer, epochs=100)
test_model(model, test_loader)

Epoch 1/100, Loss: 1.2530
Epoch 2/100, Loss: 1.1408
Epoch 3/100, Loss: 1.1202
Epoch 4/100, Loss: 1.1163
Epoch 5/100, Loss: 1.0958
Epoch 6/100, Loss: 1.0747
Epoch 7/100, Loss: 1.0831
Epoch 8/100, Loss: 1.0673
Epoch 9/100, Loss: 1.0727
Epoch 10/100, Loss: 1.0615
Epoch 11/100, Loss: 1.0529
Epoch 12/100, Loss: 1.0273
Epoch 13/100, Loss: 1.0189
Epoch 14/100, Loss: 1.0215
Epoch 15/100, Loss: 1.0022
Epoch 16/100, Loss: 1.0080
Epoch 17/100, Loss: 1.0120
Epoch 18/100, Loss: 1.0262
Epoch 19/100, Loss: 0.9870
Epoch 20/100, Loss: 0.9915
Epoch 21/100, Loss: 0.9773
Epoch 22/100, Loss: 0.9868
Epoch 23/100, Loss: 0.9785
Epoch 24/100, Loss: 0.9658
Epoch 25/100, Loss: 0.9651
Epoch 26/100, Loss: 0.9704
Epoch 27/100, Loss: 0.9558
Epoch 28/100, Loss: 0.9480
Epoch 29/100, Loss: 0.9423
Epoch 30/100, Loss: 0.9486
Epoch 31/100, Loss: 0.9360
Epoch 32/100, Loss: 0.9546
Epoch 33/100, Loss: 0.9414
Epoch 34/100, Loss: 0.9394
Epoch 35/100, Loss: 0.9284
Epoch 36/100, Loss: 0.9280
Epoch 37/100, Loss: 0.9263
Epoch 38/1

In [10]:
# 預測新數據並保存到同一個 CSV
def predict_and_save_combined(model, selected_features, files, output_csv):
    results = []
    for file_path, wine_type in files:
        data = pd.read_csv(file_path, delimiter=',')
        if selected_features:
            data = data[selected_features]
        X_new = scaler.transform(data.values)
        X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = model(X_new_tensor)
            _, predicted = torch.max(outputs, 1)

        results.extend([
            {'ID': f"{wine_type}_{i+1}", 'quality': int(pred.cpu().numpy())}
            for i, pred in enumerate(predicted)
        ])

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    print(f"Predictions saved to {output_csv}")

In [12]:
# 預測紅酒與白酒品質，合併輸出至單一 CSV
predict_and_save_combined(model,
  selected_features,
 [("winequality-red_goal.csv", "red"), ("winequality-white_goal.csv", "white")],
                          "winequality_predictions.csv")

Predictions saved to winequality_predictions.csv
